# Automatic Fix AV classification
This notebook demonstrates how to fix a predicted AV classification map using topological recontrustion of the vascular tree.

In [1]:
from pathlib import Path

import numpy as np
from jppype import Mosaic, vscode_theme

from fundus_vessels_toolkit import FundusData
from fundus_vessels_toolkit.pipelines import AVSegToTree, NaiveAVSegToTree
from fundus_vessels_toolkit.segment_to_graph.av_map_fixing import TopologicalLabel, fix_av_map, rasterize_tree_topology
from fundus_vessels_toolkit.utils.data_io import load_label_image
from fundus_vessels_toolkit.utils.jppype import draw_tree, draw_trees

vscode_theme()

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

### Load a fundus image and its AV map

In [2]:
IMG = "g_007.png"
PATH = Path("/home/gaby/These/Data/Fundus/Vessels/GAVE/training/")


# Path to the raw fundus image
RAW_PATH = PATH / "images" / IMG

# Path to the artery/vein segmentation
AV_TRUE = PATH / "av" / IMG

AV_PRED = PATH / "Task1_2" / IMG

# Path to the OD segmentation
OD_PATH = PATH / "od" / IMG

fundus_gt = FundusData(fundus=RAW_PATH, vessels=AV_TRUE, od=OD_PATH)
fundus = fundus_gt.update(vessels=load_label_image(AV_PRED, ["black", "yellow", "cyan"]))

m = Mosaic(2, cols_titles=["Predicted", "Ground Truth"], cell_height=400)
fundus.draw(view=m[0])
fundus_gt.draw(view=m[1])
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…

## Parse tree on the GT

Parse the topology of the ground truth segmentation and generate its topology map.

In [8]:
seg2tree = NaiveAVSegToTree()
trees_gt = seg2tree(fundus_gt)

topo_maps = [rasterize_tree_topology(tree, expand_labels_by=10) for tree in trees_gt]
(art_labels, art_topo), (vei_labels, vei_topo) = topo_maps

- The ``labels`` indicate to which subtree each vessel pixel belong, as well as the branching patterns to reach it.
- The ``topo`` map monotonically increases with the distance from the subtree root.

In [9]:
m = Mosaic(
    (2, 3),
    cols_titles=["VTree", "Branch labels", "Topology map"],
    rows_titles=["Art.", "Vein"],
    cell_height=400,
)
fundus_gt.draw(view=m[0, 0])
draw_tree(trees_gt[0], view=m[0, 0], artery=True, edge_labels=True)
m[0, 1].add_image(TopologicalLabel.map_to_rgb(art_labels))
m[0, 2].add_image(np.repeat(art_topo[:, :, None], 3, axis=2))
fundus_gt.draw(view=m[1, 0])
draw_tree(trees_gt[1], view=m[1, 0], artery=False, edge_labels=True)
m[1, 1].add_image(TopologicalLabel.map_to_rgb(vei_labels))
m[1, 2].add_image(np.repeat(vei_topo[:, :, None], 3, axis=2))
m

GridBox(children=(HTML(value='<span/>'), HTML(value='<h3 style="text-align: center;">VTree</h3>'), HTML(value=…

## Parse graph on the Prediction

In [5]:
trees_fixed = AVSegToTree()(fundus)

fundus_fixed = fundus.update(vessels=fix_av_map(fundus.av, trees_fixed))

m = Mosaic(2, cols_titles=["Predicted", "Corrected"], cell_height=800)
fundus.draw(view=m[0])
fundus_fixed.draw(view=m[1])

draw_trees(trees_fixed, view=m[1])
m

GridBox(children=(HTML(value='<h3 style="text-align: center;">Predicted</h3>'), HTML(value='<h3 style="text-al…